# Random Split Diagnostic Baseline

This notebook repeats the tabular baseline experiment using repeated random train/test splits.

Purpose:
- Diagnose how much easier random split is compared with temporal and station-temporal validation.
- Compare with literature settings where random split often reports higher R2.
- Keep temporal and station-temporal validation as the main research evidence.

Important interpretation:
- Random split is diagnostic only.
- Random split can place similar station/year patterns in both train and test, so it may overestimate deployment performance.
- DO and BOD are indirect proxy targets. Turbidity and SS are optical-related targets.

In [ ]:
from __future__ import annotations

import math
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    XGBRegressor = None
    HAS_XGBOOST = False
    print('XGBoost unavailable:', repr(exc))

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception as exc:
    CatBoostRegressor = None
    HAS_CATBOOST = False
    print('CatBoost unavailable:', repr(exc))

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

In [ ]:
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
PROJECT_ROOT = Path(r'E:\Water Quality Research')
DATA_PATH = PROJECT_ROOT / 'Data' / 'tabular_data_14' / 'train' / 'rswq_model_input_relaxed_s2_context_2562_2568.csv'
OUTPUT_DIR = PROJECT_ROOT / 'Reports' / 'random_split_diagnostic'

SAVE_OUTPUTS = True
RUN_STACKING = True
INCLUDE_XGBOOST = HAS_XGBOOST
INCLUDE_CATBOOST = HAS_CATBOOST

# Diagnostic random split settings.
RANDOM_SPLIT_TEST_SIZE = 0.20
RANDOM_SPLIT_SEEDS = [13, 42, 101, 202, 303]
N_STACKING_INNER_FOLDS = 3

# Keep this diagnostic focused on targets discussed in the consultation.
TARGETS_TO_RUN = ['DO', 'BOD', 'Turbidity', 'SS']
FEATURE_SETS_TO_RUN = ['S2 only', 'Context only', 'S2 + Context']

TREE_N_ESTIMATORS = 350
BOOST_N_ESTIMATORS = 450
CATBOOST_ITERATIONS = 500

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_PATH:', DATA_PATH)
print('SAVE_OUTPUTS:', SAVE_OUTPUTS)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Random seeds:', RANDOM_SPLIT_SEEDS)
print('HAS_XGBOOST:', HAS_XGBOOST, 'HAS_CATBOOST:', HAS_CATBOOST)

## Load Dataset And Define Leakage-Safe Feature Sets

Feature selection follows the same logic as the main baseline notebook:

- `S2 only`: Sentinel-2 band/index/formula median features.
- `Context only`: rainfall, ERA5-Land hourly, JRC/GSW, SRTM, WorldCover, and numeric temporal features.
- `S2 + Context`: combined feature set.

Target columns and known leakage columns are blocked from features.

In [ ]:
df = pd.read_csv(DATA_PATH)
df['year_be'] = pd.to_numeric(df['year_be'], errors='coerce')
if 'station_canonical' in df.columns:
    df['station_canonical'] = df['station_canonical'].astype(str)

REGRESSION_TARGETS = {
    'DO': {'column': 'DO_mg_l_clean', 'transform': 'raw'},
    'BOD': {'column': 'BOD_mg_l_clean', 'transform': 'raw'},
    'Turbidity': {'column': 'turbidity_NTU_clean', 'transform': 'log1p'},
    'SS': {'column': 'SS_mg_l_clean', 'transform': 'log1p'},
    # Kept here only to block leakage even though this diagnostic does not run them by default.
    'TCB': {'column': 'TCB_MPN_100ml_clean', 'transform': 'log1p'},
    'FCB': {'column': 'FCB_MPN_100ml_clean', 'transform': 'log1p'},
}

NH3_COLUMN = 'NH3_N_mg_l_clean'
ALL_TARGET_COLUMNS = [v['column'] for v in REGRESSION_TARGETS.values()] + [NH3_COLUMN]
EXTRA_LEAKAGE_COLUMNS = [
    'WQI_clean', 'WQI_reported', 'WQI_score_input_complete', 'WQI_mean_subindex_pcd5', 'WQI_recalc_pcd5',
    'pH_clean', 'conductivity_clean', 'salinity_ppt_clean', 'temp_air_clean', 'temp_water_clean',
]

s2_cols = [c for c in df.columns if c.endswith('_median')]
context_prefixes = ('rain_gsmap_', 'rainy_hours_gsmap_', 'era5h_', 'jrc_', 'srtm_', 'wc_')
context_extra_cols = ['month', 'day_of_year', 'doy_sin', 'doy_cos', 'hours_since_last_rain_gsmap_mean']
context_cols = [c for c in df.columns if c.startswith(context_prefixes)] + [c for c in context_extra_cols if c in df.columns]

numeric_cols = set(df.select_dtypes(include=[np.number]).columns)
blocked = set(ALL_TARGET_COLUMNS + EXTRA_LEAKAGE_COLUMNS)
blocked.update([c for c in df.columns if c.startswith('WQI_')])
blocked.update(['lat', 'lon', 'latitude', 'longitude'])

s2_cols = [c for c in s2_cols if c in numeric_cols and c not in blocked]
context_cols = [c for c in context_cols if c in numeric_cols and c not in blocked and c not in s2_cols]

FEATURE_SETS = {
    'S2 only': s2_cols,
    'Context only': context_cols,
    'S2 + Context': s2_cols + context_cols,
}

for name, cols in FEATURE_SETS.items():
    assert len(cols) == len(set(cols)), f'duplicate columns in {name}'
    forbidden = sorted(set(cols) & blocked)
    assert not forbidden, f'leakage columns in {name}: {forbidden}'

print('Dataset shape:', df.shape)
print('Year counts:')
display(df['year_be'].value_counts().sort_index().to_frame('n'))
print('Feature counts:')
display(pd.DataFrame({'feature_set': list(FEATURE_SETS), 'n_features': [len(v) for v in FEATURE_SETS.values()]}))
print('Targets used in this diagnostic:')
display(df[[REGRESSION_TARGETS[t]['column'] for t in TARGETS_TO_RUN]].describe().T)

## Metrics And Models

Selection metric:

- Raw targets (`DO`, `BOD`): original-scale RMSE.
- Log targets (`Turbidity`, `SS`): model-scale RMSE from the log1p target.

Original-scale metrics are still reported for every target.

In [ ]:
def safe_corr(y_true, y_pred, method='pearson'):
    y = pd.Series(y_true)
    p = pd.Series(y_pred)
    if y.nunique(dropna=True) < 2 or p.nunique(dropna=True) < 2:
        return np.nan
    return y.corr(p, method=method)


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def evaluate_predictions(y_true_original, y_pred_modelscale, transform):
    y_true_original = np.asarray(y_true_original, dtype=float)
    y_pred_modelscale = np.asarray(y_pred_modelscale, dtype=float)

    if transform == 'log1p':
        y_true_modelscale = np.log1p(y_true_original)
        y_pred_original = np.expm1(y_pred_modelscale)
        y_pred_original = np.where(np.isfinite(y_pred_original), y_pred_original, np.nan)
        y_pred_original = np.clip(y_pred_original, 0, None)
    else:
        y_true_modelscale = y_true_original
        y_pred_original = y_pred_modelscale

    ok = np.isfinite(y_true_original) & np.isfinite(y_pred_original)
    y = y_true_original[ok]
    p = y_pred_original[ok]

    metrics = {
        'original_n': len(y),
        'original_mae': mean_absolute_error(y, p),
        'original_rmse': rmse(y, p),
        'original_r2': r2_score(y, p) if len(y) > 1 else np.nan,
        'original_pearson_r': safe_corr(y, p, method='pearson'),
        'original_spearman_r': safe_corr(y, p, method='spearman'),
        'original_bias': float(np.mean(p - y)),
    }

    if transform == 'log1p':
        y_model = np.log1p(y)
        p_model = np.log1p(p)
    else:
        y_model = y
        p_model = p

    metrics.update({
        'modelscale_rmse': rmse(y_model, p_model),
        'modelscale_r2': r2_score(y_model, p_model) if len(y_model) > 1 else np.nan,
        'modelscale_pearson_r': safe_corr(y_model, p_model, method='pearson'),
        'modelscale_spearman_r': safe_corr(y_model, p_model, method='spearman'),
        'modelscale_bias': float(np.mean(p_model - y_model)),
    })
    return metrics


def make_regression_models(seed):
    models = {
        'DummyMedian': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', DummyRegressor(strategy='median')),
        ]),
        'Ridge': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('model', RidgeCV(alphas=np.logspace(-3, 3, 13))),
        ]),
        'ElasticNet': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
            ('model', ElasticNet(alpha=0.01, l1_ratio=0.2, max_iter=10000, random_state=seed)),
        ]),
        'RandomForest': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', RandomForestRegressor(n_estimators=TREE_N_ESTIMATORS, min_samples_leaf=3, random_state=seed, n_jobs=-1)),
        ]),
        'ExtraTrees': Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', ExtraTreesRegressor(n_estimators=TREE_N_ESTIMATORS, min_samples_leaf=2, random_state=seed, n_jobs=-1)),
        ]),
    }

    if INCLUDE_XGBOOST:
        models['XGBoost'] = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', XGBRegressor(
                n_estimators=BOOST_N_ESTIMATORS,
                max_depth=3,
                learning_rate=0.03,
                subsample=0.85,
                colsample_bytree=0.85,
                reg_lambda=2.0,
                objective='reg:squarederror',
                tree_method='hist',
                random_state=seed,
                n_jobs=-1,
                verbosity=0,
            )),
        ])

    if INCLUDE_CATBOOST:
        models['CatBoost'] = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('model', CatBoostRegressor(
                iterations=CATBOOST_ITERATIONS,
                depth=4,
                learning_rate=0.03,
                loss_function='RMSE',
                random_seed=seed,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1,
            )),
        ])
    return models

print('Regression models available:', list(make_regression_models(RANDOM_STATE)))

In [ ]:
def valid_regression_indices(data, target_col, indices):
    idx = pd.Index(indices)
    y = pd.to_numeric(data.loc[idx, target_col], errors='coerce')
    return idx[y.notna()].to_numpy()


def fit_predict_stacking_regression(base_models, X_train, y_train, X_test, seed):
    # Exclude DummyMedian from stacking base learners.
    stack_base = {name: model for name, model in base_models.items() if name != 'DummyMedian'}
    if len(stack_base) < 2:
        raise ValueError('Need at least two non-dummy base models for stacking.')

    kfold = KFold(n_splits=N_STACKING_INNER_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros((len(X_train), len(stack_base)), dtype=float)
    test_meta = np.zeros((len(X_test), len(stack_base)), dtype=float)

    for j, (name, model) in enumerate(stack_base.items()):
        fold_test_preds = []
        for tr_pos, val_pos in kfold.split(X_train):
            fold_model = clone(model)
            fold_model.fit(X_train.iloc[tr_pos], y_train[tr_pos])
            oof[val_pos, j] = fold_model.predict(X_train.iloc[val_pos])
            fold_test_preds.append(fold_model.predict(X_test))
        test_meta[:, j] = np.mean(fold_test_preds, axis=0)

    meta = RidgeCV(alphas=np.logspace(-3, 3, 13))
    meta.fit(oof, y_train)
    return meta.predict(test_meta)


def evaluate_random_split_once(data, seed):
    rows = []
    train_idx, test_idx = train_test_split(
        data.index.to_numpy(),
        test_size=RANDOM_SPLIT_TEST_SIZE,
        shuffle=True,
        random_state=seed,
    )

    for target_name in TARGETS_TO_RUN:
        target_info = REGRESSION_TARGETS[target_name]
        target_col = target_info['column']
        transform = target_info['transform']

        target_train_idx = valid_regression_indices(data, target_col, train_idx)
        target_test_idx = valid_regression_indices(data, target_col, test_idx)
        y_train_original = pd.to_numeric(data.loc[target_train_idx, target_col], errors='coerce').to_numpy(dtype=float)
        y_test_original = pd.to_numeric(data.loc[target_test_idx, target_col], errors='coerce').to_numpy(dtype=float)
        y_train_modelscale = np.log1p(y_train_original) if transform == 'log1p' else y_train_original

        for feature_set_name in FEATURE_SETS_TO_RUN:
            feature_cols = FEATURE_SETS[feature_set_name]
            X_train = data.loc[target_train_idx, feature_cols]
            X_test = data.loc[target_test_idx, feature_cols]
            models = make_regression_models(seed)

            for model_name, model in models.items():
                try:
                    fitted = clone(model)
                    fitted.fit(X_train, y_train_modelscale)
                    y_pred = fitted.predict(X_test)
                    metrics = evaluate_predictions(y_test_original, y_pred, transform)
                    rows.append({
                        'split': 'random',
                        'seed': seed,
                        'task': 'regression',
                        'target': target_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'transform': transform,
                        'n_features': len(feature_cols),
                        'n_train': len(target_train_idx),
                        'n_test': len(target_test_idx),
                        **metrics,
                    })
                except Exception as exc:
                    rows.append({
                        'split': 'random',
                        'seed': seed,
                        'task': 'regression',
                        'target': target_name,
                        'feature_set': feature_set_name,
                        'model': model_name,
                        'transform': transform,
                        'n_features': len(feature_cols),
                        'n_train': len(target_train_idx),
                        'n_test': len(target_test_idx),
                        'error': repr(exc),
                    })

            if RUN_STACKING:
                try:
                    y_pred_stack = fit_predict_stacking_regression(models, X_train, y_train_modelscale, X_test, seed)
                    metrics = evaluate_predictions(y_test_original, y_pred_stack, transform)
                    rows.append({
                        'split': 'random',
                        'seed': seed,
                        'task': 'regression',
                        'target': target_name,
                        'feature_set': feature_set_name,
                        'model': 'Stacking_RidgeMeta',
                        'transform': transform,
                        'n_features': len(feature_cols),
                        'n_train': len(target_train_idx),
                        'n_test': len(target_test_idx),
                        **metrics,
                    })
                except Exception as exc:
                    rows.append({
                        'split': 'random',
                        'seed': seed,
                        'task': 'regression',
                        'target': target_name,
                        'feature_set': feature_set_name,
                        'model': 'Stacking_RidgeMeta',
                        'transform': transform,
                        'n_features': len(feature_cols),
                        'n_train': len(target_train_idx),
                        'n_test': len(target_test_idx),
                        'error': repr(exc),
                    })
    return rows

## Run Repeated Random Split

This cell can take time because it evaluates multiple models, feature sets, targets, and seeds.

Set `INCLUDE_XGBOOST`, `INCLUDE_CATBOOST`, `TREE_N_ESTIMATORS`, or `RANDOM_SPLIT_SEEDS` in the configuration cell if a faster diagnostic run is needed.

In [ ]:
all_rows = []
for seed in RANDOM_SPLIT_SEEDS:
    print(f'Running random split seed={seed} ...')
    all_rows.extend(evaluate_random_split_once(df, seed))

random_results = pd.DataFrame(all_rows)
print('random_results shape:', random_results.shape)

if 'error' in random_results.columns:
    error_rows = random_results[random_results['error'].notna()]
    print('error rows:', len(error_rows))
    if len(error_rows):
        display(error_rows[['seed', 'target', 'feature_set', 'model', 'error']].head(20))

display(random_results.head())

In [ ]:
random_ok = random_results.copy()
if 'error' in random_ok.columns:
    random_ok = random_ok[random_ok['error'].isna()].copy()

random_ok['rank_metric'] = np.where(
    random_ok['transform'].eq('log1p'),
    random_ok['modelscale_rmse'],
    random_ok['original_rmse'],
)

agg_spec = {
    'n_features': 'first',
    'n_train': 'mean',
    'n_test': 'mean',
    'original_mae': ['mean', 'std'],
    'original_rmse': ['mean', 'std'],
    'original_r2': ['mean', 'std'],
    'original_pearson_r': 'mean',
    'original_spearman_r': 'mean',
    'original_bias': 'mean',
    'modelscale_rmse': ['mean', 'std'],
    'modelscale_r2': ['mean', 'std'],
    'modelscale_pearson_r': 'mean',
    'modelscale_spearman_r': 'mean',
    'modelscale_bias': 'mean',
    'rank_metric': 'mean',
}

random_agg = random_ok.groupby(['target', 'feature_set', 'model', 'transform'], as_index=False).agg(agg_spec)
random_agg.columns = ['_'.join([x for x in c if x]) if isinstance(c, tuple) else c for c in random_agg.columns]
random_agg = random_agg.rename(columns={
    'n_features_first': 'n_features',
    'n_train_mean': 'n_train_mean',
    'n_test_mean': 'n_test_mean',
})

TARGET_ORDER = {name: i for i, name in enumerate(TARGETS_TO_RUN)}
FEATURE_ORDER = {name: i for i, name in enumerate(FEATURE_SETS_TO_RUN)}
random_agg['target_order'] = random_agg['target'].map(TARGET_ORDER)
random_agg['feature_order'] = random_agg['feature_set'].map(FEATURE_ORDER)
random_agg['is_dummy'] = random_agg['model'].str.contains('Dummy', case=False, na=False)

random_best_overall = (
    random_agg[~random_agg['is_dummy']]
    .sort_values(['target_order', 'rank_metric_mean', 'original_rmse_mean'])
    .groupby('target', as_index=False)
    .head(1)
)

random_best_by_feature = (
    random_agg[~random_agg['is_dummy']]
    .sort_values(['target_order', 'feature_order', 'rank_metric_mean', 'original_rmse_mean'])
    .groupby(['target', 'feature_set'], as_index=False)
    .head(1)
)

show_cols = [
    'target', 'feature_set', 'model', 'transform', 'n_features', 'n_train_mean', 'n_test_mean',
    'original_mae_mean', 'original_rmse_mean', 'original_r2_mean', 'original_spearman_r_mean',
    'original_bias_mean', 'modelscale_rmse_mean', 'modelscale_r2_mean', 'modelscale_spearman_r_mean',
]

print('Best overall random split models, averaged across seeds')
display(random_best_overall[show_cols])

print('Best random split models by feature set, averaged across seeds')
display(random_best_by_feature[show_cols].sort_values(['target_order', 'feature_order']))

## Compare Against Temporal And Station-Temporal Results

This section loads curated result tables from `reports/tables` when the notebook is run from the repository.

If those files are unavailable, the random-split diagnostic still runs normally.

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    candidates = [start] + list(start.parents)
    known = Path(r'C:\Users\ADMIN\Documents\Codex\2026-07-09\new-chat-2\Water-Quality-Assessment')
    candidates.append(known)
    for cand in candidates:
        if (cand / 'reports' / 'tables').exists() and (cand / 'README.md').exists():
            return cand
    return None

repo_root = find_repo_root()
report_table_dir = repo_root / 'reports' / 'tables' if repo_root else None
print('repo_root:', repo_root)
print('report_table_dir:', report_table_dir)

comparison_rows = []

# Random split best rows.
for _, row in random_best_overall.iterrows():
    comparison_rows.append({
        'target': row['target'],
        'validation': 'random split diagnostic',
        'feature_set': row['feature_set'],
        'model': row['model'],
        'transform': row['transform'],
        'original_rmse': row['original_rmse_mean'],
        'original_r2': row['original_r2_mean'],
        'original_spearman_r': row['original_spearman_r_mean'],
        'modelscale_rmse': row['modelscale_rmse_mean'],
        'modelscale_r2': row['modelscale_r2_mean'],
        'modelscale_spearman_r': row['modelscale_spearman_r_mean'],
    })

if report_table_dir and report_table_dir.exists():
    manual_temporal_path = report_table_dir / 'manual_temporal_regression_all_configs.csv'
    autogluon_path = report_table_dir / 'autogluon_regression_all_configs.csv'
    station_temporal_path = report_table_dir / 'manual_station_temporal_regression_best_by_target_feature.csv'

    temporal_frames = []
    if manual_temporal_path.exists():
        m = pd.read_csv(manual_temporal_path)
        m = m[m['target'].isin(TARGETS_TO_RUN)].copy()
        m['source'] = 'manual temporal'
        temporal_frames.append(m)
    if autogluon_path.exists():
        a = pd.read_csv(autogluon_path)
        a = a[a['target'].isin(TARGETS_TO_RUN)].copy()
        a['source'] = 'autogluon temporal'
        temporal_frames.append(a)

    if temporal_frames:
        temporal = pd.concat(temporal_frames, ignore_index=True)
        temporal['rank_metric'] = np.where(temporal['transform'].eq('log1p'), temporal['modelscale_rmse'], temporal['original_rmse'])
        temporal['target_order'] = temporal['target'].map(TARGET_ORDER)
        temporal_best = temporal.sort_values(['target_order', 'rank_metric', 'original_rmse']).groupby('target', as_index=False).head(1)
        for _, row in temporal_best.iterrows():
            comparison_rows.append({
                'target': row['target'],
                'validation': row.get('source', 'temporal'),
                'feature_set': row['feature_set'],
                'model': row['model'],
                'transform': row['transform'],
                'original_rmse': row['original_rmse'],
                'original_r2': row['original_r2'],
                'original_spearman_r': row['original_spearman_r'],
                'modelscale_rmse': row['modelscale_rmse'],
                'modelscale_r2': row['modelscale_r2'],
                'modelscale_spearman_r': row['modelscale_spearman_r'],
            })

    if station_temporal_path.exists():
        st = pd.read_csv(station_temporal_path)
        st = st[st['target'].isin(TARGETS_TO_RUN)].copy()
        st['target_order'] = st['target'].map(TARGET_ORDER)
        st_best = st.sort_values(['target_order', 'selection_rmse_mean']).groupby('target', as_index=False).head(1)
        for _, row in st_best.iterrows():
            comparison_rows.append({
                'target': row['target'],
                'validation': 'station-temporal robustness',
                'feature_set': row['feature_set'],
                'model': row['model'],
                'transform': row['transform'],
                'original_rmse': row['original_rmse_mean'],
                'original_r2': row['original_r2_mean'],
                'original_spearman_r': row['original_spearman_r_mean'],
                'modelscale_rmse': row['modelscale_rmse_mean'],
                'modelscale_r2': row['modelscale_r2_mean'],
                'modelscale_spearman_r': row['modelscale_spearman_r_mean'],
            })

comparison_df = pd.DataFrame(comparison_rows)
if not comparison_df.empty:
    comparison_df['target_order'] = comparison_df['target'].map(TARGET_ORDER)
    display(comparison_df.sort_values(['target_order', 'validation']))
else:
    print('No comparison table could be built.')

In [ ]:
# Plot validation comparison if available.
if 'comparison_df' in globals() and not comparison_df.empty:
    plot_df = comparison_df.copy()
    plot_df['reported_r2'] = np.where(plot_df['transform'].eq('log1p'), plot_df['modelscale_r2'], plot_df['original_r2'])
    plot_df = plot_df.sort_values(['target_order', 'validation'])

    fig, ax = plt.subplots(figsize=(11, 5))
    labels = plot_df['target'] + '\n' + plot_df['validation'].str.replace(' ', '\n')
    ax.bar(np.arange(len(plot_df)), plot_df['reported_r2'])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(np.arange(len(plot_df)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylabel('R2 (model-scale for log targets)')
    ax.set_title('Random Split vs Temporal vs Station-Temporal Validation')
    plt.tight_layout()
    plt.show()

## Save Diagnostic Outputs

Outputs are saved outside the Git repository by default:

`E:\Water Quality Research\Reports\random_split_diagnostic`

In [ ]:
if SAVE_OUTPUTS:
    random_results.to_csv(OUTPUT_DIR / 'random_split_regression_all_rows.csv', index=False, encoding='utf-8-sig')
    random_agg.to_csv(OUTPUT_DIR / 'random_split_regression_aggregated.csv', index=False, encoding='utf-8-sig')
    random_best_overall.to_csv(OUTPUT_DIR / 'random_split_regression_best_overall.csv', index=False, encoding='utf-8-sig')
    random_best_by_feature.to_csv(OUTPUT_DIR / 'random_split_regression_best_by_feature_set.csv', index=False, encoding='utf-8-sig')
    if 'comparison_df' in globals() and not comparison_df.empty:
        comparison_df.to_csv(OUTPUT_DIR / 'random_vs_temporal_station_temporal_comparison.csv', index=False, encoding='utf-8-sig')
    print('Saved outputs to:', OUTPUT_DIR)

## Interpretation Guide

Use this diagnostic carefully:

- If random split R2 is much higher than temporal/station-temporal R2, the model may be interpolating familiar station/year patterns.
- Random split can be useful for comparison with literature, but it should not replace temporal and station-temporal validation.
- For Turbidity and SS, model-scale R2 and Spearman correlation are often more informative than original-scale R2 when the target is skewed.
- For DO and BOD, results should be described as proxy prediction rather than direct satellite measurement.

Suggested reporting sentence:

`Random split was used as a diagnostic comparison with common literature settings. The main validation remains temporal and station-temporal testing because those settings better evaluate future-year and unseen-station generalization.`